# От confounder clusters до sensitivity protocol

Первая функция предлагает наблюдаемые **benchmark groups**. Вторая использует их, чтобы оценить устойчивость DML estimate к **hidden confounding**.

## 1. Аргументы и результат

### `rank_confounder_clusters(data, effect_estimation, ...)`

Возвращает **DataFrame** с одной строкой на cluster. `features` — список его признаков, `cluster_id` — ID до сортировки. `importance_d` и `importance_y` — агрегированные feature importances для treatment и outcome. `score` — их произведение. Для ranking достаточно сохранённых diagnostics.

| Аргумент | Что задаёт |
|---|---|
| `data`, `effect_estimation` | `causaldata` с колонками confounders и `dml_result` из fitted IRM с native feature importance. |
| `min_abs_correlation=0.5` | Порог абсолютной correlation в примере. |
| `correlation_method="pearson"`, `linkage_method="complete"` | Pearson correlation и complete linkage. В каждом cluster порог выполняется для каждой пары признаков. |
| `max_samples=200_000`, `random_state=42` | Лимит строк и seed для подвыборки при расчёте correlation. |
| `outcome_importance="mean"` | Среднее importance из `g0` и `g1` по умолчанию. Есть вариант `"max"`. |
| `ranking_method="mixed"` | По умолчанию первые две группы идут по `score`, третья — по `importance_d` среди оставшихся. `"joint"` сортирует все группы по `score`. |

### `run_sensitivity_protocol(effect_estimation, data, ...)`

Возвращает **dict**. `status` и `summary` дают итог, `primary`, `stress` и `adversarial` — таблицы scenarios. `benchmarks` содержит калибровку по каждому refit, `details` — расчёты по группе и scenario, `warnings` — пояснения к диагностике.

| Аргумент | Что задаёт |
|---|---|
| `effect_estimation`, `data` | Исходные `dml_result` и `causaldata` с тем же порядком строк, что при fit. |
| `benchmark_groups` | Словарь «имя группы → список confounders». На каждую группу приходится отдельный short-model refit. |
| `decision_threshold=0.0` | Порог в единицах outcome. Для требования «эффект больше 5» задайте `5.0` и `direction="positive"`. |
| `direction="auto"` | `positive` при исходной оценке на пороге или выше, иначе `negative`. Для заранее заданного directional claim укажите направление явно. |
| `preconditions_passed=True` | Внешний результат проверок adjustment set, overlap, nuisance models и стабильности estimate. В примере считаем, что они пройдены. |
| `alpha=0.05`, `stress_multiplier=2.0`, `fit_args=None` | Уровень для bias-aware CI, множитель для `stress` и опциональные аргументы для `estimate()` в short models. |

## 2. Запуск и разбор output

Используем [`generate_obs_hte_26_rich`](../research/generate_obs_hte_26_rich.ipynb) с исходными 100 000 строками и `seed=42`. DGP задаёт 11 confounders и heterogeneous treatment effects. Оцениваем **ATTE** — средний эффект среди treated. Если `causaldata` и `dml_result` у вас уже есть, перейдите к следующей code cell.

In [1]:
from catboost import CatBoostClassifier, CatBoostRegressor

from causalis.scenarios.unconfoundedness import IRM
from causalis.scenarios.unconfoundedness.dgp import generate_obs_hte_26_rich

causaldata = generate_obs_hte_26_rich(
    n=100_000, seed=42, include_oracle=False, return_causal_data=True,
)
learner_options = dict(
    iterations=100, depth=5, random_seed=42, thread_count=2,
    verbose=False, allow_writing_files=False,
)
model = IRM(
    ml_g=CatBoostRegressor(**learner_options),
    ml_m=CatBoostClassifier(**learner_options),
    n_folds=3,
    random_state=42,
).fit(causaldata, store_diagnostics=True)
dml_result = model.estimate(score="ATTE")

In [2]:
from causalis.shared import rank_confounder_clusters
from causalis.scenarios.unconfoundedness.refutation import run_sensitivity_protocol

ranking = rank_confounder_clusters(
    causaldata,
    dml_result,
    min_abs_correlation=0.5,
    correlation_method="pearson",
    linkage_method="complete",
    max_samples=200_000,
    random_state=42,
)

top3 = ranking.head(3)
benchmark_groups = {
    f"rank_{rank}_cluster_{row['cluster_id']}": list(row["features"])
    for rank, (_, row) in enumerate(top3.iterrows(), start=1)
}

protocol = run_sensitivity_protocol(
    dml_result,
    causaldata,
    benchmark_groups=benchmark_groups,
    decision_threshold=0.0,
    direction="auto",
    preconditions_passed=True,
)

In [3]:
ranking_preview = ranking[["cluster_id", "features", "importance_d", "importance_y", "score"]].head(5)
ranking_preview = ranking_preview.set_axis(range(1, len(ranking_preview) + 1))
display(ranking_preview.rename_axis("rank").round(4))
print(benchmark_groups)

,cluster_id,features,importance_d,importance_y,score
rank,,,,,
1,6,[support_tickets_90d],0.1215,0.2025,0.0246
2,1,[avg_sessions_week],0.0717,0.2657,0.0191
3,2,[spend_last_month],0.1842,0.0588,0.0108
4,0,[tenure_months],0.1476,0.1102,0.0163
5,4,[income_monthly],0.1495,0.0881,0.0132


{'rank_1_cluster_6': ['support_tickets_90d'], 'rank_2_cluster_1': ['avg_sessions_week'], 'rank_3_cluster_2': ['spend_last_month']}


На этой выборке максимальная абсолютная Pearson correlation между разными признаками около $0.40$. При пороге $0.50$ все clusters состоят из одного признака. У `spend_last_month` `score` ниже, чем у `tenure_months`, но выше `importance_d`. Поэтому `mixed` ставит `spend_last_month` третьим.

In [4]:
print(f"{dml_result.estimand} (long model) = {dml_result.value:.3f}")
print(protocol["status"])
print(protocol["summary"])
print(f"limited_margin = {protocol['limited_margin']}")

display(protocol["benchmarks"][[
    "benchmark", "theta_short", "cf_y", "cf_d", "rho", "boundary_calibration",
]].round(4))

display(protocol["scenarios"][[
    "benchmark", "scenario", "ci_lower", "ci_upper", "passed",
]].round(3))

for warning in protocol["warnings"]:
    print(warning)

ATTE (long model) = 13.427
PASS
PASS: the practically meaningful effect is robust to hidden confounding comparable to every primary benchmark. Every primary bias-aware confidence interval satisfies the positive decision rule (ci_lower > 0). The reported stress-test margin is limited. DoubleML boundary calibration was used for ['rank_3_cluster_2'].
limited_margin = True


,benchmark,theta_short,cf_y,cf_d,rho,boundary_calibration
0,rank_1_cluster_6,5.4652,0.3743,0.0077,-0.1747,False
1,rank_2_cluster_1,12.5979,0.3883,0.0038,-0.0253,False
2,rank_3_cluster_2,12.6459,0.0418,0.0000,-1.0000,True


,benchmark,scenario,ci_lower,ci_upper,passed
0,rank_1_cluster_6,primary,0.944,25.955,True
1,rank_2_cluster_1,primary,8.156,18.703,True
2,rank_3_cluster_2,primary,8.986,17.868,True
3,rank_1_cluster_6,2x,-7.179,34.121,False
4,rank_2_cluster_1,2x,7.324,19.539,True
5,rank_3_cluster_2,2x,8.986,17.868,True
6,rank_1_cluster_6,adversarial,-38.034,65.094,False
7,rank_2_cluster_1,adversarial,-24.506,51.523,False
8,rank_3_cluster_2,adversarial,8.986,17.868,True


direction='auto' inferred 'positive' from theta=13.4271 relative to decision_threshold=0; use an explicit direction for a pre-specified directional claim.
Benchmark 'rank_3_cluster_2' used DoubleML boundary calibration: cf_d_raw=-0.00056214 was clipped to 0; rho used the DoubleML boundary fallback because both gain components were not strictly positive.
IRM currently uses n_rep=1; repeated cross-fitting stability must be assessed separately.


- **`benchmarks`** показывает, что произошло при исключении группы. Без `support_tickets_90d` оценка снизилась с $13.43$ до $5.47$. Для `spend_last_month` отрицательный raw gain по Riesz moment обрезался до нуля. Это видно по `cf_d=0`, `boundary_calibration=True` и в `warnings`.
- **`primary`** использует калиброванные силы и $\rho$. У всех трёх групп `ci_lower > 0`, поэтому общий статус `PASS`.
- **`stress`** (строки `2x`) усиливает benchmark. **`adversarial`** сохраняет силы и задаёт $\rho=1$. У `support_tickets_90d` нижняя граница падает с $0.94$ в `primary` до $-7.18$ и $-38.03$ соответственно. Поэтому `limited_margin=True`.

## 3. Математика

### Как получается ranking

Для двух признаков $i,j$ считаем $d_{ij}=1-|\operatorname{corr}(X_i,X_j)|$. **Complete linkage** измеряет расстояние между clusters по самой далёкой паре, $d(A,B)=\max_{i\in A,j\in B}d_{ij}$. Cut на высоте $1-0.5=0.5$ оставляет только группы, где каждая пара проходит порог.

Обрезаем отрицательные native importances до нуля и нормируем отдельно для $m$, $g_0$, $g_1$, чтобы в каждой колонке сумма была $1$. $m$ предсказывает treatment, $g_0$ и $g_1$ — outcome для control и treated. Для группы $C$ получаем

$$
I_D(C)=\sum_{j\in C}I_{m,j},\quad
I_Y(C)=\tfrac{1}{2}\sum_{j\in C}(I_{g_0,j}+I_{g_1,j}),\quad
\mathrm{score}(C)=I_D(C)I_Y(C).
$$

### Как получается bias-aware CI

Для каждой группы сравниваем **long model** со всеми confounders и **short model** без этой группы. Данные и cross-fitting folds совпадают. Сравниваем effect estimate $\theta$, mean squared outcome residual $\sigma^2$ и Riesz moment $\nu^2$. Последний связан с весами outcome residuals в DML score. Силы `cf_y` и `cf_d` получаем из изменений этих двух moments, а $\rho$ — из сдвига effect estimate.

$$
G_y=\sigma_S^2-\sigma_L^2,\quad G_d=\nu_L^2-\nu_S^2,\qquad
cf_{y,\mathrm{raw}}=\frac{G_y}{\sigma_L^2},\quad
cf_{d,\mathrm{raw}}=\frac{G_d}{\nu_S^2}.
$$

При $G_y,G_d>0$ берём $\rho_{\mathrm{raw}}=(\theta_S-\theta_L)/\sqrt{G_yG_d}$. Ограничиваем $\rho$ диапазоном $[-1,1]$, а $cf_y,cf_d$ — диапазоном $[0,1]$. При неположительном gain соответствующая сила равна $0$, а $\rho=\operatorname{sign}(\theta_S-\theta_L)$. Сила, достигшая $1$, делает calibration invalid.

Для `stress` умножаем $cf_y$ и odds $cf_d/(1-cf_d)$ на $k$. Получаем $cf_y'=\min(k\,cf_y,1-10^{-12})$ и $cf_d'=k\,cf_d/(1-cf_d+k\,cf_d)$. Для `adversarial` подставляем $\rho=1$.

$$
B=|\rho|\sqrt{\sigma_L^2\nu_L^2\,cf_y\frac{cf_d}{1-cf_d}},
\qquad
\mathrm{bias\text{-}aware\ CI}
=\left[\theta_L-B-q\,\mathrm{SE}_{-},\;\theta_L+B+q\,\mathrm{SE}_{+}\right].
$$

$B$ — сдвиг от центра до одной границы bias bound. $\mathrm{SE}_{-}$ и $\mathrm{SE}_{+}$ учитывают sampling uncertainty каждой границы. В iid-примере при `alpha=0.05` квантиль $q\approx1.96$. Для `positive` требуется `ci_lower > threshold`, для `negative` — `ci_upper < threshold`. Равенство порогу даёт `False`.

## 4. Псевдокод

```text
distance = 1 - abs(correlation(confounders))
clusters = cut(complete_linkage(distance), height=1 - min_abs_correlation)
ranking = order_by_mixed_rule(clusters, normalized_feature_importances)
groups = features_from_first_3_rows(ranking)

для каждой группы:
    short = refit_irm(исходные данные, без признаков группы, те же folds)
    cf_y, cf_d, rho = calibrate(long, short)
    для scenario в [primary, stress, adversarial]:
        ci = bias_aware_ci(long, параметры scenario)
        passed[группа, scenario] = scenario valid и ci строго за threshold

PASS = preconditions_passed и все primary passed
limited_margin = PASS и хотя бы один stress или adversarial не прошёл
```